In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH, "data", "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

os.makedirs(TABLE_PATH, exist_ok=True)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase4_eda.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Dataset restored.")
print("Rows:", len(feature_df))
print("Stations:", feature_df["StationId"].nunique())

In [ ]:
pollutant_columns = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

print("Pollutants:", pollutant_columns)

In [ ]:
target_source = feature_df[
    ["StationId", "Datetime", "AQI"]
].copy()

target_source["TargetDatetime"] = (
    target_source["Datetime"]
    + pd.Timedelta(hours=1)
)

target_source = target_source.rename(
    columns={
        "AQI": "AQI_t_plus_1"
    }
)

feature_df = feature_df.merge(
    target_source[
        [
            "StationId",
            "TargetDatetime",
            "AQI_t_plus_1"
        ]
    ],
    left_on=["StationId", "Datetime"],
    right_on=["StationId", "TargetDatetime"],
    how="left",
    validate="many_to_one"
)

feature_df = feature_df.drop(
    columns=["TargetDatetime"]
)

print(
    "Rows with exact t+1 AQI:",
    feature_df["AQI_t_plus_1"].notna().sum()
)

print(
    "Rows without exact t+1 AQI:",
    feature_df["AQI_t_plus_1"].isna().sum()
)

In [ ]:
before_target_filter = len(feature_df)

feature_df = feature_df.dropna(
    subset=["AQI_t_plus_1"]
).copy()

after_target_filter = len(feature_df)

print("Rows before:", before_target_filter)
print("Rows after:", after_target_filter)
print(
    "Removed because exact t+1 was unavailable:",
    before_target_filter - after_target_filter
)

In [ ]:
target_check = feature_df[
    [
        "StationId",
        "StationName",
        "Datetime",
        "AQI",
        "AQI_t_plus_1"
    ]
].copy()

target_check["TargetTime"] = (
    target_check["Datetime"]
    + pd.Timedelta(hours=1)
)

print(
    "Target alignment check:",
    (
        target_check["TargetTime"]
        == target_check["TargetTime"]
    ).all()
)

display(target_check.head(10))

In [ ]:
lag_hours = [1, 2, 12]

lag_base = feature_df[
    ["StationId", "Datetime", "AQI"] + pollutant_columns
].copy()

for lag in lag_hours:
    lag_source = lag_base.copy()

    lag_source["Datetime"] = (
        lag_source["Datetime"]
        + pd.Timedelta(hours=lag)
    )

    rename_map = {
        column: f"{column}_lag_{lag}h"
        for column in ["AQI"] + pollutant_columns
    }

    lag_source = lag_source.rename(
        columns=rename_map
    )

    feature_df = feature_df.merge(
        lag_source,
        on=["StationId", "Datetime"],
        how="left",
        validate="one_to_one"
    )

print("Exact timestamp-based lag features created.")

In [ ]:
lag_columns = [
    column
    for column in feature_df.columns
    if "_lag_" in column
]

display(
    feature_df[
        ["StationId", "Datetime", "AQI"] + lag_columns
    ].head(10)
)

In [ ]:
# Remove any rolling columns created by the failed attempt
existing_rolling_columns = [
    col for col in feature_df.columns
    if "rolling_" in col
]

feature_df = feature_df.drop(
    columns=existing_rolling_columns,
    errors="ignore"
)

# Columns for rolling statistics
rolling_base_columns = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2"
]

rolling_parts = []

# Process each station independently
for station_id, group in feature_df[
    ["StationId", "Datetime"] + rolling_base_columns
].groupby("StationId"):

    group = group.sort_values("Datetime").copy()

    # Use Datetime as the time index
    group = group.set_index("Datetime")

    rolling_result = pd.DataFrame(index=group.index)

    for column in rolling_base_columns:

        # 3-hour rolling mean
        rolling_result[f"{column}_rolling_mean_3h"] = (
            group[column]
            .rolling("3h", min_periods=1)
            .mean()
        )

        # 3-hour rolling standard deviation
        rolling_result[f"{column}_rolling_std_3h"] = (
            group[column]
            .rolling("3h", min_periods=2)
            .std()
        )

        # 12-hour rolling mean
        rolling_result[f"{column}_rolling_mean_12h"] = (
            group[column]
            .rolling("12h", min_periods=1)
            .mean()
        )

        # 12-hour rolling standard deviation
        rolling_result[f"{column}_rolling_std_12h"] = (
            group[column]
            .rolling("12h", min_periods=2)
            .std()
        )

    rolling_result["StationId"] = station_id
    rolling_result["Datetime"] = rolling_result.index

    rolling_parts.append(
        rolling_result.reset_index(drop=True)
    )

# Combine all stations
rolling_features_df = pd.concat(
    rolling_parts,
    ignore_index=True
)

# Safety check for duplicate merge keys
duplicate_keys = rolling_features_df.duplicated(
    subset=["StationId", "Datetime"]
).sum()

print(
    "Duplicate StationId + Datetime keys in rolling data:",
    duplicate_keys
)

# Remove duplicates only if any unexpected duplicates remain
rolling_features_df = (
    rolling_features_df
    .drop_duplicates(
        subset=["StationId", "Datetime"],
        keep="first"
    )
)

# Merge safely
feature_df = feature_df.merge(
    rolling_features_df,
    on=["StationId", "Datetime"],
    how="left",
    validate="one_to_one"
)

print("Rolling features created successfully.")
print("Dataset shape:", feature_df.shape)

In [ ]:
rolling_columns = [
    col for col in feature_df.columns
    if "rolling_" in col
]

print("Number of rolling features:", len(rolling_columns))
print(rolling_columns)

display(
    feature_df[
        ["StationId", "Datetime", "AQI"]
        + rolling_columns
    ].head(10)
)

In [ ]:
duplicate_main_keys = feature_df.duplicated(
    subset=["StationId", "Datetime"]
).sum()

print(
    "Duplicate StationId + Datetime records:",
    duplicate_main_keys
)

In [ ]:
feature_df["Hour"] = feature_df["Datetime"].dt.hour
feature_df["Day"] = feature_df["Datetime"].dt.day
feature_df["Month"] = feature_df["Datetime"].dt.month
feature_df["Year"] = feature_df["Datetime"].dt.year
feature_df["DayOfWeek"] = feature_df["Datetime"].dt.dayofweek

feature_df["IsWeekend"] = (
    feature_df["DayOfWeek"] >= 5
).astype("int8")

print(
    feature_df[
        [
            "Datetime",
            "Hour",
            "Day",
            "Month",
            "Year",
            "DayOfWeek",
            "IsWeekend"
        ]
    ].head()
)

In [ ]:
def assign_season(month):
    if month in [10, 11]:
        return "PostMonsoon"
    elif month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Other"

feature_df["Season"] = (
    feature_df["Month"]
    .apply(assign_season)
)

display(
    feature_df["Season"].value_counts()
)

In [ ]:
feature_df["IsStubbleSeason"] = (
    feature_df["Month"]
    .isin([10, 11])
).astype("int8")

print(
    feature_df["IsStubbleSeason"].value_counts()
)

In [ ]:
diwali_dates = pd.to_datetime([
    "2015-11-11",
    "2016-10-30",
    "2017-10-19",
    "2018-11-07",
    "2019-10-27",
    "2020-11-14"
])

diwali_dates

In [ ]:
def is_diwali_window(timestamp, dates, days_before=2, days_after=2):
    return any(
        (timestamp >= date - pd.Timedelta(days=days_before))
        and (timestamp <= date + pd.Timedelta(days=days_after))
        for date in dates
    )

feature_df["IsDiwaliWindow"] = feature_df[
    "Datetime"
].apply(
    lambda x: int(
        is_diwali_window(
            x,
            diwali_dates,
            days_before=2,
            days_after=2
        )
    )
)

print(
    feature_df["IsDiwaliWindow"].value_counts()
)

In [ ]:
feature_df["StationId"] = (
    feature_df["StationId"]
    .astype("string")
)

print(
    "Unique stations:",
    feature_df["StationId"].nunique()
)

In [ ]:
temporary_columns = [
    column
    for column in feature_df.columns
    if column.endswith("_was_missing")
]

if "AQI_extreme_flag" in feature_df.columns:
    temporary_columns.append(
        "AQI_extreme_flag"
    )

feature_df = feature_df.drop(
    columns=temporary_columns,
    errors="ignore"
)

print("Removed temporary columns:", temporary_columns)

In [ ]:
current_features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

lag_features = [
    column
    for column in feature_df.columns
    if "_lag_" in column
]

rolling_features = [
    column
    for column in feature_df.columns
    if "rolling_" in column
]

calendar_features = [
    "Hour",
    "Day",
    "Month",
    "Year",
    "DayOfWeek",
    "IsWeekend",
    "Season"
]

event_features = [
    "IsStubbleSeason",
    "IsDiwaliWindow"
]

spatial_features = [
    "StationId"
]

print("Current features:", len(current_features))
print("Lag features:", len(lag_features))
print("Rolling features:", len(rolling_features))
print("Calendar features:", len(calendar_features))
print("Event features:", len(event_features))
print("Spatial features:", len(spatial_features))

In [ ]:
candidate_features = (
    current_features
    + lag_features
    + rolling_features
)

engineered_missing = (
    feature_df[candidate_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(
    engineered_missing[
        engineered_missing > 0
    ].to_frame("MissingPercentage")
)

In [ ]:
before_final_target_filter = len(feature_df)

feature_df = feature_df.dropna(
    subset=["AQI_t_plus_1"]
).copy()

print(
    "Rows removed:",
    before_final_target_filter - len(feature_df)
)

print(
    "Rows remaining:",
    len(feature_df)
)

In [ ]:
forbidden_future_columns = [
    "AQI_t_plus_1"
]

leakage_columns = [
    column
    for column in candidate_features
    if column in forbidden_future_columns
]

print(
    "Future-target columns found in features:",
    leakage_columns
)

In [ ]:
validation_target = feature_df[
    ["StationId", "Datetime", "AQI_t_plus_1"]
].copy()

validation_target["ExpectedTargetTime"] = (
    validation_target["Datetime"]
    + pd.Timedelta(hours=1)
)

# Use the original timestamp-target relationship to verify
target_lookup = feature_df[
    ["StationId", "Datetime", "AQI"]
].rename(
    columns={
        "Datetime": "ExpectedTargetTime",
        "AQI": "ExpectedAQI"
    }
)

validation_target = validation_target.merge(
    target_lookup,
    on=["StationId", "ExpectedTargetTime"],
    how="left",
    validate="one_to_one"
)

mismatch_count = (
    ~np.isclose(
        validation_target["AQI_t_plus_1"],
        validation_target["ExpectedAQI"],
        equal_nan=True
    )
).sum()

print(
    "Target alignment mismatches:",
    mismatch_count
)

In [ ]:
feature_dataset_path = os.path.join(
    PROCESSED_PATH,
    "delhi_feature_engineered.parquet"
)

feature_df.to_parquet(
    feature_dataset_path,
    index=False
)

print("Feature-engineered dataset saved:")
print(feature_dataset_path)

In [ ]:
feature_inventory = {
    "target": ["AQI_t_plus_1"],
    "current_features": current_features,
    "lag_features": lag_features,
    "rolling_features": rolling_features,
    "calendar_features": calendar_features,
    "event_features": event_features,
    "spatial_features": spatial_features
}

feature_inventory_path = os.path.join(
    OUTPUT_PATH,
    "feature_inventory.json"
)

import json

with open(
    feature_inventory_path,
    "w"
) as f:
    json.dump(
        feature_inventory,
        f,
        indent=4
    )

print("Feature inventory saved:")
print(feature_inventory_path)

In [ ]:
print("=" * 60)
print("PHASE 5 — FEATURE ENGINEERING SUMMARY")
print("=" * 60)

print("Final rows:", len(feature_df))
print("Stations:", feature_df["StationId"].nunique())
print("Target:", "AQI_t_plus_1")

print("\nFeature counts:")
print("Current:", len(current_features))
print("Lag:", len(lag_features))
print("Rolling:", len(rolling_features))
print("Calendar:", len(calendar_features))
print("Events:", len(event_features))
print("Spatial:", len(spatial_features))

print(
    "\nTarget missing:",
    feature_df["AQI_t_plus_1"].isna().sum()
)

print(
    "Target negative:",
    (feature_df["AQI_t_plus_1"] < 0).sum()
)

print("=" * 60)

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save Phase 5 feature-engineered dataset
# --------------------------------------------------

phase5_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase5_feature_engineered.parquet"
)

feature_df.to_parquet(
    phase5_checkpoint,
    index=False
)

# --------------------------------------------------
# 2. Save feature inventory
# --------------------------------------------------

feature_inventory = {
    "target": ["AQI_t_plus_1"],
    "current_features": current_features,
    "lag_features": lag_features,
    "rolling_features": rolling_features,
    "calendar_features": calendar_features,
    "event_features": event_features,
    "spatial_features": spatial_features
}

feature_inventory_path = os.path.join(
    OUTPUT_PATH,
    "feature_inventory.json"
)

with open(
    feature_inventory_path,
    "w"
) as f:
    json.dump(
        feature_inventory,
        f,
        indent=4
    )

# --------------------------------------------------
# 3. Save Phase 5 summary
# --------------------------------------------------

phase5_summary = {
    "completed_phase": "Phase 5 - Feature Engineering",
    "next_phase": "Phase 6 - Leakage-Free Baseline and Model Training",

    "rows": int(len(feature_df)),
    "stations": int(feature_df["StationId"].nunique()),

    "target": "AQI_t_plus_1",

    "current_feature_count": len(current_features),
    "lag_feature_count": len(lag_features),
    "rolling_feature_count": len(rolling_features),
    "calendar_feature_count": len(calendar_features),
    "event_feature_count": len(event_features),
    "spatial_feature_count": len(spatial_features),

    "target_missing": int(
        feature_df["AQI_t_plus_1"].isna().sum()
    ),

    "target_negative": int(
        (feature_df["AQI_t_plus_1"] < 0).sum()
    )
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase5_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase5_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 4. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 5 - Feature Engineering",
    "next_phase": "Phase 6 - Leakage-Free Baseline and Model Training",
    "dataset": "delhi_phase5_feature_engineered.parquet"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Verify saved files
# --------------------------------------------------

files_to_check = [
    phase5_checkpoint,
    feature_inventory_path,
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 5 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 5")
print("Next: Phase 6 - Leakage-Free Baseline and Model Training")